### 🛠️ Phase 1: Delta Encoding & Decoding

Delta encoding is a lossy-to-lossless preprocessing technique widely used in data compression, especially effective for continuous data streams or gradient images.

#### 📌 Core Principle
Instead of storing absolute values, the encoder records the **difference (delta) between the current byte and the previous byte**.
* **Encoding Formula**: $D_i = (X_i - X_{i-1}) \pmod{256}$
* **Decoding Formula**: $X_i = (D_i + X_{i-1}) \pmod{256}$

#### 🌟 Design Highlight: Modular Arithmetic Defense
* **Overflow/Underflow Prevention**: By applying the `% 256` modulo operation, the data is **guaranteed to stay perfectly within the unsigned 8-bit byte range (`0 ~ 255`)** even when subtractions yield negative numbers or additions overflow. This completely eliminates hardware-level overflow bugs.

#### 🎯 Primary Objective
This preprocessing transforms varying data streams into **large clusters of repeating `0`s or uniform values**, drastically reducing the information entropy and smoothing the path for the subsequent LZ77 algorithm.

In [1]:
def delta_encode(data: bytes) -> bytearray:
    """Transforms continuous data into incremental byte-level differences (Deltas).
    
    This preprocessing layout reduces entropy by generating clusters of zeros,
    which optimizes the data stream for subsequent LZ77 window matching.
    """
    if not data:
        return bytearray()
    
    output = bytearray(len(data))
    output[0] = data[0] 
    
    for i in range(1, len(data)):
        # 🌟 WHY: Using modular arithmetic (% 256) ensures the delta wraps around seamlessly.
        # This keeps the result strictly within the unsigned 8-bit range (0-255) and prevents integer underflow.
        output[i] = (data[i] - data[i-1]) % 256
    return output

def delta_decode(data: bytearray) -> bytes:
    """Reconstructs the original byte stream from modulo-encoded delta differences."""
    if not data:
        return b""
    
    output = bytearray(len(data))
    output[0] = data[0]
    
    for i in range(1, len(data)):
        # 🌟 WHY: Reverses the encoding phase. Modulo 256 automatically resolves any previous underflow wraps.
        output[i] = (data[i] + output[i-1]) % 256
    return bytes(output)

### 📦 Phase 2: LZ77 Sliding Window Compression

This phase implements a Sliding Window dictionary-based compression algorithm, utilizing a Hash Map to accelerate repeated string matching.

#### ⚙️ Strict Boundary & Parameter Constraints
To ensure a seamless handoff to the downstream binary bit-packing stage, parameters are strictly capped to prevent bit-width overflow:
* **Search Window Size**: Rigidly locked at `3000` bytes. This ensures the offset value fits comfortably under the 12-bit maximum threshold of 4095, acting as a boundary defense.
* **Lookahead Buffer**: The maximum match length is strictly capped at `255` bytes, aligning perfectly with the 8-bit upper limit.

#### 🚀 Performance Optimization
* **Triple-Hash Acceleration**: A dictionary (`pos_hash`) tracks historical byte positions using a 3-byte tuple (`triple`) as the key. This drastically reduces the search complexity from $O(N^2)$ window scanning to near-constant lookups.

#### 📄 Output Token Format
The data stream is converted into a list of structurally unified tokens:
1. **Literal Token (No Match)**: `(False, literal_value, 0)`
2. **Reference Token (Match)**: `(True, distance, length)`

In [2]:
def lz77_compress(data: bytes, window_size: int = 3000) -> list:
    """Compresses data using a sliding window dictionary approach.
    
    Returns a list of structured tokens:
    - Literal: (False, byte_value, 0)
    - Reference Match: (True, distance, length)
    """
    tokens = []
    cursor = 0
    data_len = len(data)
    max_match_len = 255  # 🌟 WHY: Capped at 255 to perfectly fit into a single downstream 8-bit stream segment.
    
    pos_hash = {}
    
    while cursor < data_len:
        match_dist = 0
        match_len = 0
        
        if cursor + 3 <= data_len:
            triple = (data[cursor], data[cursor+1], data[cursor+2])
            p = pos_hash.get(triple, -1)
            
            # 🌟 WHY: Enforces boundary controls. The historical position must reside inside the active 3000-byte window.
            if p != -1 and (cursor - p <= window_size) and (p < cursor):
                curr_match_len = 0
                while (cursor + curr_match_len < data_len and \
                       data[p + curr_match_len] == data[cursor + curr_match_len] and \
                       curr_match_len < max_match_len):
                    curr_match_len += 1
                    
                if curr_match_len >= 3:
                    match_len = curr_match_len
                    match_dist = cursor - p

        if match_len >= 3:
            # 🌟 WHY: Defensive check to ensure values do not exceed the architectural limits of 12-bit/8-bit bitstreams.
            if 0 < match_dist <= 4095 and 3 <= match_len <= 255:
                tokens.append((True, match_dist, match_len))
                if cursor + 3 <= data_len:
                    triple = (data[cursor], data[cursor+1], data[cursor+2])
                    pos_hash[triple] = cursor
                cursor += match_len
                continue
                
        tokens.append((False, data[cursor], 0))
        if cursor + 3 <= data_len:
            triple = (data[cursor], data[cursor+1], data[cursor+2])
            pos_hash[triple] = cursor
        cursor += 1
            
    return tokens

def lz77_decompress(tokens: list) -> bytes:
    """Restores the raw byte sequences from LZ77 literal and reference tokens."""
    output = bytearray()
    for is_match, val, length in tokens:
        if not is_match:
            output.append(val)
        else:
            distance = val
            start_pos = len(output) - distance
            for i in range(length):
                output.append(output[start_pos + i])
    return bytes(output)

### 🛠️ Phase 3: Hybrid Compression Pipeline with Adaptive Tree Selection

This phase implements a highly optimized, dual-pass hybrid compression pipeline that dynamically adapts its serialization model based on empirical entropy evaluation. The system chains Delta Encoding, Hash-Accelerated LZ77 Sliding Window string substitution, and Canonical Huffman Entropy Coding into a deterministic, high-efficiency data reduction pipeline.

#### 📌 Core Principles & Mathematical Foundation

The compression engine operates by minimizing the total bit length of the compressed archive, denoted as $L_{\text{total}}$:

$$L_{\text{total}} = \sum_{i \in \text{Symbols}} (\text{freq}_i \cdot \text{len}_i) + L_{\text{header}}$$

Where $\text{freq}_i$ represents the symbol frequency and $\text{len}_i$ represents the calculated bit-width under the Canonical Huffman mapping. Rather than forcing a static architectural model, the engine conducts a parallel pre-encoding cost simulation between two distinct modes to resolve the structural trade-off between header bloat and payload density:

1. **Mode 0 (Single Consolidated Tree)**: Consolidates literals ($0 \sim 255$), the End-of-Stream (EOS) boundary marker ($256$), and an LZ77 Match Flag ($257$) into a unified alphabet. Symbol $257$ triggers static down-stream hardware bit-packing widths ($12\text{-bit}$ distance, $8\text{-bit}$ length).
2. **Mode 1 (Dual Split Trees)**: Implements independent entropy contexts. **Tree A** maps literals and match lengths via an alphabet upper-bounded at $512$. **Tree B** maps dictionary lookup distances via an alphabet upper-bounded at $4096$, serializing distance vectors into variable-length huffman codes.

The canonical representation guarantees lexicographical code assignment using only symbol bit-lengths, calculating the base numeric code $C_L$ for any length $L$ via:

$$C_L = (C_{L-1} + \text{Count}_{L-1}) \ll 1$$

#### ⚙️ Boundary & Parameter Constraints

* **Bitstream Packaging**: The `BitWriter` and `BitReader` operate on standard big-endian, MSB-first bitwise serialization layout. Bit accumulation flushes seamlessly on strict $8\text{-bit}$ byte-aligned boundaries.
* **Sliding Window Safeguards**: Dictionary back-references are constrained to a maximum distance of $3000\text{ bytes}$ ($\le 12\text{-bit}$ limit of $4095$) and matching run lengths are strictly bounded at $255\text{ bytes}$ ($8\text{-bit}$ unsigned limit).
* **Header Allocation Contract**: Mode 1 introduces a dynamic upper-bound truncation technique where unused trailing elements in the code length tables are completely omitted from the layout, dramatically slashing header overhead on sparse datasets.

#### 🌟 Design Highlights & Defensive Architecture

* **Intelligent Optimization Decision**: The system performs a dual-pass estimation of the exact byte size for both Mode 0 and Mode 1. It acts as an active compile-time gatekeeper, routing small or high-entropy files (e.g., small `.txt` files) through Mode 0 to prevent header-induced file expansion, while routing predictable, repetitive blocks (e.g., `.bmp` images) through Mode 1.
* **Bitwise Corruption Cascades**: The `BitWriter` enforces explicit input clamping via `value & ((1 << num_bits) - 1)` to prevent overlapping memory corruption during malformed downstream serialization.
* **Out-of-Bounds Memory Protection**: Decompression vectors execute defensive tracking guards, such as collapsing out-of-bound historical lookups via `if distance > len(output): distance = len(output)` and gracefully handling zero-frequency constraints during decoding to neutralize buffer-overflow attack vectors.

#### 📄 Binary Archive Layout Specification

The output format contract creates a crystalline layout structured as follows:

| Component | Size (Bytes) | Protocol Description |
| :--- | :--- | :--- |
| **Magic Header** | 2 Bytes | Fixed signature format bytes `b"MY"` |
| **Original Size** | 4 Bytes | Unsigned 32-bit integer (`<I`, Little-endian) tracking baseline byte size |
| **Mode Flag** | 1 Byte | `0` (Single Tree Consolidated) or `1` (Dual Split Tree Architecture) |
| **Tree Structures** | Variable | **Mode 0**: Fixed 258-byte length table.<br>**Mode 1**: Truncated 2-byte upper bounds ($max\_lit\_len, max\_dist$) followed by localized byte arrays. |
| **Bitstream Payload** | Variable | Bitstream serialized MSB-first, closed via explicit EOS token ($256$). |

In [3]:
import struct
import heapq
from collections import Counter
from typing import Dict, List, Tuple, Optional, Any

class BitWriter:
    """Low-latency bitstream sequencer responsible for packing variable-length 
    codes into tightly aligned byte arrays.
    """

    def __init__(self) -> None:
        self.bytes_data = bytearray()
        self.buffer = 0
        self.bit_count = 0

    def write_bits(self, value: int, num_bits: int) -> None:
        """Writes an integer value using a specific bit-width into the buffer,
        automatically flushing to the byte array upon reaching byte boundaries.

        Args:
            value (int): The integer value to be written.
            num_bits (int): The precise bit-width allocation allocated for the value.
        """
        # 🛡️ DEFENSE: Force bitmasking to isolate value range and prevent overlapping corruption
        clean_value = value & ((1 << num_bits) - 1)
        
        for i in range(num_bits - 1, -1, -1):
            bit = (clean_value >> i) & 1
            self.buffer = (self.buffer << 1) | bit
            self.bit_count += 1
            
            # 🌟 WHY: Commit to byte array exactly at 8-bit boundaries to release CPU register pressure
            if self.bit_count == 8:
                self.bytes_data.append(self.buffer)
                self.buffer = 0
                self.bit_count = 0

    def flush(self) -> bytearray:
        """Forces the serialization of remaining bits in the buffer, left-shifting
        and zero-padding to guarantee byte-alignment.

        Returns:
            bytearray: The completed, byte-aligned binary structure.
        """
        if self.bit_count > 0:
            # 🌟 WHY: Left-shift remaining bits to align with standard stream parsing direction
            padding = 8 - self.bit_count
            self.buffer = self.buffer << padding
            self.bytes_data.append(self.buffer)
            self.buffer = 0
            self.bit_count = 0
        return self.bytes_data


class BitReader:
    """Stream decompression decoder embedded with out-of-bounds memory protection cascades."""

    def __init__(self, data: bytes) -> None:
        self.data = data
        self.byte_idx = 0
        self.bit_idx = 7

    def read_bit(self) -> int:
        """Extracts a single bit from the binary stream.

        Returns:
            int: 0 or 1. If EOF is breached, it defaults to 0 as an internal defensive fallback.
        """
        # 🛡️ DEFENSE: Abort index translation if binary stream is corrupted or truncated
        if self.byte_idx >= len(self.data):
            return 0
            
        bit = (self.data[self.byte_idx] >> self.bit_idx) & 1
        self.bit_idx -= 1
        
        if self.bit_idx < 0:
            self.bit_idx = 7
            self.byte_idx += 1
        return bit

    def read_bits(self, num_bits: int) -> int:
        """Reads multiple bits from the stream and reconstructs them into an integer.

        Args:
            num_bits (int): Total count of sequential bits to extract.

        Returns:
            int: The reconstructed integer value.
        """
        value = 0
        for _ in range(num_bits):
            value = (value << 1) | self.read_bit()
        return value


def generate_canonical_codes(code_lengths: Dict[int, int]) -> Dict[int, str]:
    """Generates a deterministic Canonical Huffman codebook directly from symbol 
    code lengths without needing tree structure traversal.

    Args:
        code_lengths (Dict[int, int]): Mapping of active symbols to their Huffman bit-lengths.

    Returns:
        Dict[int, str]: Mapping of symbols to their explicit binary string representations.
    """
    if not code_lengths:
        return {}
        
    max_len = max(code_lengths.values())
    bl_count = Counter(code_lengths.values())
    
    # 🌟 WHY: Establish the sequential Lexicographical Base code value for each bit-length rank
    next_code = {}
    code = 0
    bl_count[0] = 0
    for bits in range(1, max_len + 1):
        code = (code + bl_count[bits - 1]) << 1
        next_code[bits] = code
        
    # 🌟 WHY: Distribute codes monotonically by sorted symbol index to ensure perfect determinism
    canonical_codes = {}
    for sym in sorted(code_lengths.keys()):
        length = code_lengths[sym]
        if length > 0:
            code_val = next_code[length]
            canonical_codes[sym] = format(code_val, f'0{length}b')
            next_code[length] += 1
            
    return canonical_codes


def get_huffman_lengths(frequencies: Dict[int, int], max_symbols: int) -> Dict[int, int]:
    """Computes targeted code path lengths for all symbols using a min-heap Huffman implementation.

    Args:
        frequencies (Dict[int, int]): Statistical frequency distribution of the symbols.
        max_symbols (int): Static output dimension boundary used for data contract serialization alignment.

    Returns:
        Dict[int, int]: Code lengths for all potential symbols (unused symbols are tracked as 0).
    """
    # 🛡️ DEFENSE: Prevent min-heap initialization failures on completely empty inputs
    if len(frequencies) == 0: 
        return {i: 0 for i in range(max_symbols)}
        
    # 🛡️ DEFENSE: Isolate uniform dataset edge cases, assigning a single bit for protocol compliance
    if len(frequencies) == 1:
        sym = list(frequencies.keys())[0]
        return {i: (1 if i == sym else 0) for i in range(max_symbols)}
        
    # 🌟 WHY: Use a Min-Heap layout to construct the bottom-up tree combination in O(N log N) time
    heap = [[wt, sym, [sym]] for sym, wt in frequencies.items()]
    heapq.heapify(heap)
    
    lengths = {sym: 0 for sym in frequencies}
    while len(heap) > 1:
        lo = heapq.heappop(heap)
        hi = heapq.heappop(heap)
        # 🌟 WHY: Increment depth tracking for all leaf descendants belonging to the merged subtrees
        for sym in lo[2]: lengths[sym] += 1
        for sym in hi[2]: lengths[sym] += 1
        heapq.heappush(heap, [lo[0] + hi[0], min(lo[1], hi[1]), lo[2] + hi[2]])
        
    # 🌟 WHY: Pad unreferenced symbols with 0 to comply with fixed serialization array contracts
    full_lengths = {i: 0 for i in range(max_symbols)}
    for sym, l in lengths.items():
        full_lengths[sym] = l
    return full_lengths


def my_custom_compress(original_data: bytes) -> bytes:
    """Orchestrates the hybrid compression stream, running dynamic cost evaluations 
    to decide the optimal serialization layout before emitting the binary payload.

    Args:
        original_data (bytes): Input byte stream.

    Returns:
        bytes: Compressed binary blob sequence.
    """
    if not original_data:
        return b""
    
    # Pre-processing stage: Eliminate first-order entropy and duplicate sequences
    delta_data = delta_encode(original_data)
    lz77_tokens = lz77_compress(delta_data)
    
    # ------------------ Simulation: Mode 0 (Single Consolidated Tree) ------------------
    m0_frequencies = Counter()
    for is_match, val, length in lz77_tokens:
        if not is_match:
            m0_frequencies[val] += 1
        else:
            m0_frequencies[257] += 1  # 257 denotes an LZ77 reference tag
    m0_frequencies[256] += 1  # 256 acts as the End-of-Stream (EOF) boundary marker
    
    m0_lengths = get_huffman_lengths(m0_frequencies, 258)
    
    m0_bit_payload = 0
    for is_match, val, length in lz77_tokens:
        if not is_match:
            m0_bit_payload += m0_lengths[val]
        else:
            # 🌟 WHY: In Mode 0, the tag has a variable code length, but downstream tokens use fixed bit allocations
            m0_bit_payload += m0_lengths[257] + 12 + 8 
    m0_bit_payload += m0_lengths[256]
    # Equation: Base Header(6B) + Mode Flag(1B) + Fixed Tree Table(258B) + Bitstream Payload
    m0_total_estimate = 6 + 1 + 258 + ((m0_bit_payload + 7) // 8)
    
    # ------------------ Simulation: Mode 1 (Dual Split Trees) ------------------
    m1_lit_len_freqs = Counter()
    m1_dist_freqs = Counter()
    for is_match, val, length in lz77_tokens:
        if not is_match:
            m1_lit_len_freqs[val] += 1
        else:
            len_sym = 257 + (length - 3)  # Shift match length into the upper symbol range
            m1_lit_len_freqs[len_sym] += 1
            m1_dist_freqs[val] += 1
    m1_lit_len_freqs[256] += 1 
    
    m1_lit_len_lengths = get_huffman_lengths(m1_lit_len_freqs, 512)
    m1_dist_lengths = get_huffman_lengths(m1_dist_freqs, 4096)
    
    # 🌟 WHY: Locate maximum active bounds to safely truncate headers, avoiding storing empty tail entries
    max_lit_len_used = max([i for i, l in m1_lit_len_lengths.items() if l > 0] + [257])
    max_dist_used = max([i for i, l in m1_dist_lengths.items() if l > 0] + [0])
    
    m1_bit_payload = 0
    for is_match, val, length in lz77_tokens:
        if not is_match:
            m1_bit_payload += m1_lit_len_lengths[val]
        else:
            len_sym = 257 + (length - 3)
            m1_bit_payload += m1_lit_len_lengths[len_sym] + m1_dist_lengths[val]
    m1_bit_payload += m1_lit_len_lengths[256]
    # Equation: Base Header(6B) + Mode(1B) + TreeA_Bound(2B) + TreeA + TreeB_Bound(2B) + TreeB + Payload
    m1_total_estimate = 6 + 1 + 2 + (max_lit_len_used + 1) + 2 + (max_dist_used + 1) + ((m1_bit_payload + 7) // 8)

    # ------------------ 🌟 Intelligent Optimization Decision ------------------
    chosen_mode = 0 if m0_total_estimate <= m1_total_estimate else 1
    
    # ------------------ Binary Pack Preparation ------------------
    header = bytearray()
    header += struct.pack("<2sI", b"MY", len(original_data)) 
    header.append(chosen_mode) 
    
    writer = BitWriter()
    
    if chosen_mode == 0:
        for sym in range(258):
            header.append(m0_lengths[sym])
            
        huff_table = generate_canonical_codes({k: v for k, v in m0_lengths.items() if v > 0})
        for is_match, val, length in lz77_tokens:
            if not is_match:
                bit_str = huff_table[val]
                writer.write_bits(int(bit_str, 2), len(bit_str))
            else:
                bit_str = huff_table[257]
                writer.write_bits(int(bit_str, 2), len(bit_str))
                writer.write_bits(val, 12)   # Store 12-bit history offset
                writer.write_bits(length, 8)  # Store 8-bit duplicate span
        
        end_str = huff_table[256]
        writer.write_bits(int(end_str, 2), len(end_str))
        
    else:
        header += struct.pack("<H", max_lit_len_used)
        for i in range(max_lit_len_used + 1):
            header.append(m1_lit_len_lengths[i])
            
        header += struct.pack("<H", max_dist_used)
        for i in range(max_dist_used + 1):
            header.append(m1_dist_lengths[i])
            
        lit_len_table = generate_canonical_codes({k: v for k, v in m1_lit_len_lengths.items() if v > 0})
        dist_table = generate_canonical_codes({k: v for k, v in m1_dist_lengths.items() if v > 0})
        
        for is_match, val, length in lz77_tokens:
            if not is_match:
                bit_str = lit_len_table[val]
                writer.write_bits(int(bit_str, 2), len(bit_str))
            else:
                len_sym = 257 + (length - 3)
                bit_str_len = lit_len_table[len_sym]
                writer.write_bits(int(bit_str_len, 2), len(bit_str_len))
                bit_str_dist = dist_table[val]
                writer.write_bits(int(bit_str_dist, 2), len(bit_str_dist))
                
        end_str = lit_len_table[256]
        writer.write_bits(int(end_str, 2), len(end_str))
        
    return bytes(header) + writer.flush()


def my_custom_decompress(compressed_bytes: bytes) -> bytes:
    """Parses binary headers, re-maps Canonical tree sets, and expands token vectors 
    back to the baseline state.

    Args:
        compressed_bytes (bytes): Complete raw compressed archive bytes.

    Returns:
        bytes: Fully restored uncompressed data byte stream.
    """
    if not compressed_bytes:
        return b""
        
    magic, orig_size = struct.unpack("<2sI", compressed_bytes[:6])
    if magic != b"MY":
        raise ValueError("Invalid compressed archive format signature.")
        
    chosen_mode = compressed_bytes[6]
    idx = 7
    output = bytearray()
    
    if chosen_mode == 0:
        code_lengths = {}
        for sym in range(258):
            length = compressed_bytes[idx]
            if length > 0: code_lengths[sym] = length
            idx += 1
            
        # 🌟 WHY: Invert the mapping to allow highly efficient O(1) hash lookups during bitwise sequence processing
        reverse_table = {bits: sym for sym, bits in generate_canonical_codes(code_lengths).items()}
        reader = BitReader(compressed_bytes[idx:])
        
        curr_bits = ""
        while True:
            curr_bits += str(reader.read_bit())
            if curr_bits in reverse_table:
                sym = reverse_table[curr_bits]
                curr_bits = ""
                
                if sym == 256: 
                    break  
                elif sym <= 255: 
                    output.append(sym)
                elif sym == 257:
                    distance = reader.read_bits(12)
                    length = reader.read_bits(8)
                    
                    # 🛡️ DEFENSE: Clamp distance referencing to prevent lookback out-of-bounds history violations
                    if distance > len(output): distance = len(output)
                    if distance == 0:
                        for _ in range(length): output.append(0)
                        continue
                        
                    start_pos = len(output) - distance
                    for _ in range(length):
                        output.append(output[start_pos])
                        start_pos += 1
                        
    else:
        max_lit_len_used = struct.unpack("<H", compressed_bytes[idx:idx+2])[0]
        idx += 2
        lit_len_lengths = {i: 0 for i in range(512)}
        for i in range(max_lit_len_used + 1):
            lit_len_lengths[i] = compressed_bytes[idx]
            idx += 1
        reverse_lit_len_table = {bits: sym for sym, bits in generate_canonical_codes(lit_len_lengths).items()}
        
        max_dist_used = struct.unpack("<H", compressed_bytes[idx:idx+2])[0]
        idx += 2
        dist_lengths = {i: 0 for i in range(4096)}
        for i in range(max_dist_used + 1):
            dist_lengths[i] = compressed_bytes[idx]
            idx += 1
        reverse_dist_table = {bits: sym for sym, bits in generate_canonical_codes(dist_lengths).items()}
        
        reader = BitReader(compressed_bytes[idx:])
        
        curr_bits = ""
        while True:
            curr_bits += str(reader.read_bit())
            if curr_bits in reverse_lit_len_table:
                sym = reverse_lit_len_table[curr_bits]
                curr_bits = ""
                
                if sym == 256: 
                    break
                elif sym <= 255: 
                    output.append(sym)
                elif sym >= 257:
                    # 🌟 WHY: In our mapping protocol, literal values >= 257 correspond directly to coded LZ77 length matches
                    length = (sym - 257) + 3
                    dist_bits = ""
                    while True:
                        dist_bits += str(reader.read_bit())
                        if dist_bits in reverse_dist_table:
                            distance = reverse_dist_table[dist_bits]
                            break
                            
                    # 🛡️ DEFENSE: Sliding window safety verification to eliminate buffer overflow attack vectors
                    if distance > len(output): distance = len(output)
                    if distance == 0:
                        for _ in range(length): output.append(0)
                        continue
                        
                    start_pos = len(output) - distance
                    for _ in range(length):
                        output.append(output[start_pos])
                        start_pos += 1
                        
    return delta_decode(output)

In [4]:
import os
import time
import datetime
import numpy as np
import pandas as pd
from collections import Counter

# ==========================================
# EXPERIMENT CONFIGURATION
# ==========================================
# Change these variables every time you upgrade your algorithm
CURRENT_VERSION = "v4_dynamic_optimized"
VERSION_NOTES = "Version 4: Optimized pipeline architecture featuring adaptive tree selection."

# Benchmark settings
HISTORY_FILE = "benchmark_history.csv"
NUM_RUNS = 5  # Number of runs to average out timing noise
TARGET_FILES = [
    "test1.txt",
    "test2.txt",
    "test3.txt",
    "Lenna.bmp",
    "Cameraman.bmp"
]

def run_benchmark_with_history():
    """
    Runs the benchmark for the current version, persists the data into a CSV file,
    and displays both the current run and historical comparison.
    """
    current_results = []
    timestamp = datetime.datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    
    print(f"🚀 Starting Benchmark for Version: {CURRENT_VERSION}")
    print(f"📝 Notes: {VERSION_NOTES}")
    print(f"🔄 Repeating each test {NUM_RUNS} times to eliminate timing noise...\n")
    
    for file_name in TARGET_FILES:
        if not os.path.exists(file_name):
            print(f"⚠️ Warning: '{file_name}' not found. Skipped.")
            continue
            
        # 1. Read original data
        with open(file_name, "rb") as f:
            original_data = f.read()
            
        orig_size = len(original_data)
        if orig_size == 0:
            continue
            
        # 2. Benchmark Compression (Run multiple times for averaging)
        comp_times = []
        compressed_data = b""
        for _ in range(NUM_RUNS):
            t0 = time.perf_counter()
            compressed_data = my_custom_compress(original_data)
            t1 = time.perf_counter()
            comp_times.append((t1 - t0) * 1000) # Convert to ms
            
        comp_time_ms = np.mean(comp_times)
        comp_size = len(compressed_data)
        
        # 3. Benchmark Decompression (Run multiple times for averaging)
        decomp_times = []
        decompressed_data = b""
        for _ in range(NUM_RUNS):
            t2 = time.perf_counter()
            decompressed_data = my_custom_decompress(compressed_data)
            t3 = time.perf_counter()
            decomp_times.append((t3 - t2) * 1000) # Convert to ms
            
        decomp_time_ms = np.mean(decomp_times)
        
        # 4. Calculate Objective Metrics
        comp_ratio = orig_size / comp_size if comp_size > 0 else 0
        space_saving = ((orig_size - comp_size) / orig_size) * 100
        
        # Throughput in MB/s = (Bytes / 1024 / 1024) / (ms / 1000)
        orig_size_mb = orig_size / (1024 * 1024)
        comp_throughput = orig_size_mb / (comp_time_ms / 1000) if comp_time_ms > 0 else 0
        decomp_throughput = orig_size_mb / (decomp_time_ms / 1000) if decomp_time_ms > 0 else 0
        
        is_valid = "PASS" if decompressed_data == original_data else "FAIL"
        
        # 5. Store current run data
        current_results.append({
            "timestamp": timestamp,
            "version": CURRENT_VERSION,
            "notes": VERSION_NOTES,
            "file_name": file_name,
            "orig_size_bytes": orig_size,
            "comp_size_bytes": comp_size,
            "comp_time_ms": round(comp_time_ms, 2),
            "decomp_time_ms": round(decomp_time_ms, 2),
            "comp_ratio": round(comp_ratio, 2),
            "space_saving_pct": round(space_saving, 2),
            "comp_throughput_mbs": round(comp_throughput, 2),
            "decomp_throughput_mbs": round(decomp_throughput, 2),
            "verification": is_valid
        })

    # Create DataFrame for the current run
    df_current = pd.DataFrame(current_results)
    
    # ==========================================
    # PERSISTENCE (SAVE TO CSV)
    # ==========================================
    if os.path.exists(HISTORY_FILE):
        df_history = pd.read_csv(HISTORY_FILE)
        # Prevent appending duplicate entries if the block is re-run with the same version on the exact same files
        # We drop existing logs for the same version and file to keep the history clean
        df_history = df_history[~((df_history["version"] == CURRENT_VERSION) & (df_history["file_name"].isin(df_current["file_name"])))]
        df_new_history = pd.concat([df_history, df_current], ignore_index=True)
    else:
        df_new_history = df_current

    df_new_history.to_csv(HISTORY_FILE, index=False)
    print(f"💾 Successfully saved and updated results in '{HISTORY_FILE}'.")
    
    # ==========================================
    # DISPLAY 1: Current Run Report (Formatted)
    # ==========================================
    print("\n📊 --- CURRENT RUN REPORT ---")
    display_df = df_current.copy()
    display_df["orig_size_bytes"] = display_df["orig_size_bytes"].map("{:,}".format)
    display_df["comp_size_bytes"] = display_df["comp_size_bytes"].map("{:,}".format)
    display_df["comp_ratio"] = display_df["comp_ratio"].map("{:.2f}x".format)
    display_df["space_saving_pct"] = display_df["space_saving_pct"].map("{:.2f}%".format)
    display(display_df[[
        "file_name", "orig_size_bytes", "comp_size_bytes", 
        "comp_time_ms", "decomp_time_ms", "comp_ratio", 
        "space_saving_pct", "comp_throughput_mbs", "decomp_throughput_mbs", "verification"
    ]])
    
    # ==========================================
    # DISPLAY 2: Cross-Version Historical Comparison
    # ==========================================
    print("\n📈 --- HISTORICAL VERSION COMPARISON (Pivot Table) ---")
    # Reload full history to ensure data integrity
    df_full_history = pd.read_csv(HISTORY_FILE)
    
    # Pivot table to compare Space Saving (%) across versions for each file
    pivot_space = df_full_history.pivot_table(
        index="file_name", 
        columns="version", 
        values="space_saving_pct"
    )
    
    # Sort index to match target files order for consistency
    existing_targets = [f for f in TARGET_FILES if f in pivot_space.index]
    pivot_space = pivot_space.reindex(existing_targets)
    
    print("\n[Metric: Space Saving (%)] -> Higher is better. Negative means file expansion.")
    display(pivot_space.style.format("{:.2f}%").highlight_max(axis=1, color="lightgreen"))

# Execute the benchmark
run_benchmark_with_history()

🚀 Starting Benchmark for Version: v4_dynamic_optimized
📝 Notes: Version 4: Optimized pipeline architecture featuring adaptive tree selection.
🔄 Repeating each test 5 times to eliminate timing noise...

💾 Successfully saved and updated results in 'benchmark_history.csv'.

📊 --- CURRENT RUN REPORT ---


,file_name,orig_size_bytes,comp_size_bytes,comp_time_ms,decomp_time_ms,comp_ratio,space_saving_pct,comp_throughput_mbs,decomp_throughput_mbs,verification
0,test1.txt,35,287,0.42,0.10,0.12x,-720.00%,0.08,0.33,PASS
1,test2.txt,"2,638","1,993",3.57,2.92,1.32x,24.45%,0.71,0.86,PASS
2,test3.txt,"5,349","3,866",6.59,5.47,1.38x,27.72%,0.77,0.93,PASS
3,Lenna.bmp,"263,224","182,923",330.18,324.07,1.44x,30.51%,0.76,0.77,PASS
4,Cameraman.bmp,"66,616","48,648",82.12,88.49,1.37x,26.97%,0.77,0.72,PASS



📈 --- HISTORICAL VERSION COMPARISON (Pivot Table) ---

[Metric: Space Saving (%)] -> Higher is better. Negative means file expansion.


version,v1_base,v2_base,v3_two_tree,v4_dynamic_optimized
file_name,,,,
test1.txt,-3962.86%,-717.14%,-728.57%,-720.00%
test2.txt,-201.40%,24.49%,-49.89%,24.45%
test3.txt,-103.12%,27.74%,-16.88%,27.72%
Lenna.bmp,14.40%,18.80%,30.51%,30.51%
Cameraman.bmp,-2.51%,14.77%,26.97%,26.97%
